# Lab 5 — Decision Trees for Classification

**What this lab covers:**
1. How decision trees partition the feature space (rectangular boundaries)
2. Effect of hyperparameters: `max_depth`, `min_samples_split`, `min_samples_leaf`
3. Feature importance
4. Why decision trees don't need feature scaling
5. Comparing decision trees, kNN, and logistic regression
6. Evaluation metrics beyond accuracy: precision, recall, F1
7. GridSearchCV with different scoring metrics

---

### Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_moons, make_blobs
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

---
## Part 1: Visualising Decision Boundaries

Decision trees split the feature space with axis-parallel lines,
creating rectangular regions. This helper function plots those regions.

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary", ax=None):
    """Plot the decision boundary of a fitted 2D classifier."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))

    # Predict class for every point on the grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', edgecolors='black', s=20)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

### 1.1 Decision tree on make_moons (non-linear data)

In [ ]:
# make_moons: two interleaving half-circles — not linearly separable
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

tree_moons = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_moons.fit(X_moons, y_moons)

plot_decision_boundary(tree_moons, X_moons, y_moons,
                       title='Decision Tree on make_moons (max_depth=5)')
plt.show()
print(f"Training Accuracy: {tree_moons.score(X_moons, y_moons):.3f}")
# Notice: boundaries are always rectangles (axis-parallel splits)

### 1.2 Decision tree on make_blobs (4 classes)

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=400, centers=4, n_features=2,
                              cluster_std=1.5, random_state=42)

tree_blobs = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_blobs.fit(X_blobs, y_blobs)

plot_decision_boundary(tree_blobs, X_blobs, y_blobs,
                       title='Decision Tree on make_blobs (4 classes, max_depth=5)')
plt.show()
print(f"Training Accuracy: {tree_blobs.score(X_blobs, y_blobs):.3f}")

### 1.3 Shallow tree (max_depth=2) — less complex boundary

In [ ]:
tree_shallow = DecisionTreeClassifier(max_depth=2, random_state=42)
tree_shallow.fit(X_moons, y_moons)

plot_decision_boundary(tree_shallow, X_moons, y_moons,
                       title='Shallow Tree (max_depth=2)')
plt.show()
print(f"Training Accuracy: {tree_shallow.score(X_moons, y_moons):.3f}")
# Fewer splits → simpler boundary → may underfit

---
## Part 2: Decision Tree Hyperparameters

Three main controls for tree complexity:
- **`max_depth`**: how deep the tree can grow
- **`min_samples_split`**: minimum samples needed to split a node (higher = simpler)
- **`min_samples_leaf`**: minimum samples required in each leaf (higher = simpler)

### 2.1 Effect of max_depth

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

depths = [1, 2, 5, 10, 20]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

train_scores = []
test_scores = []

for i, depth in enumerate(depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    tr = tree.score(X_train, y_train)
    te = tree.score(X_test, y_test)
    train_scores.append(tr)
    test_scores.append(te)

    plot_decision_boundary(tree, X_train, y_train,
                           title=f'max_depth={depth}\nTrain: {tr:.3f}, Test: {te:.3f}',
                           ax=axes[i])

# Summary in last subplot
axes[-1].plot(depths, train_scores, 'bo-', label='Train')
axes[-1].plot(depths, test_scores, 'ro-', label='Test')
axes[-1].set_xlabel('max_depth')
axes[-1].set_ylabel('Accuracy')
axes[-1].set_title('Train vs Test Accuracy')
axes[-1].legend()
axes[-1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Overfitting = train accuracy much higher than test accuracy
# The gap widens as depth increases

### 2.2 Effect of min_samples_split

In [ ]:
min_splits = [2, 10, 30, 50]
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, min_split in enumerate(min_splits):
    tree = DecisionTreeClassifier(max_depth=10, min_samples_split=min_split, random_state=42)
    tree.fit(X_train, y_train)

    tr = tree.score(X_train, y_train)
    te = tree.score(X_test, y_test)

    plot_decision_boundary(tree, X_train, y_train,
                           title=f'min_samples_split={min_split}\nTrain: {tr:.3f}, Test: {te:.3f}',
                           ax=axes[i])

plt.tight_layout()
plt.show()
# Higher min_samples_split → fewer splits → smoother boundary

### 2.3 Effect of min_samples_leaf

In [ ]:
min_leafs = [1, 5, 15, 30]
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, min_leaf in enumerate(min_leafs):
    tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=min_leaf, random_state=42)
    tree.fit(X_train, y_train)

    tr = tree.score(X_train, y_train)
    te = tree.score(X_test, y_test)

    plot_decision_boundary(tree, X_train, y_train,
                           title=f'min_samples_leaf={min_leaf}\nTrain: {tr:.3f}, Test: {te:.3f}',
                           ax=axes[i])

plt.tight_layout()
plt.show()
# Higher min_samples_leaf → each leaf must cover more points → simpler tree

---
## Part 3: Feature Importance and Scaling

### 3.1 Feature importance
After fitting, `tree.feature_importances_` tells you how much each feature
contributed to the splits.

In [ ]:
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_blobs, y_blobs)

importances = tree.feature_importances_

plt.figure(figsize=(8, 5))
plt.bar(['Feature 1', 'Feature 2'], importances)
plt.ylabel('Importance')
plt.title('Feature Importance')
plt.ylim([0, 1])
plt.show()

for i, imp in enumerate(importances):
    print(f"Feature {i+1}: {imp:.3f}")

### 3.2 Decision trees don't need feature scaling

Trees split on thresholds (`feature > value`). They only care about
the ordering of values, not their magnitude. Scaling changes magnitude
but not ordering, so it has zero effect on trees.

kNN and logistic regression, on the other hand, use distances/gradients
and are heavily affected by scale.

In [ ]:
# Create data with wildly different feature scales
np.random.seed(42)
X_unscaled = np.column_stack([
    np.random.normal(0, 1, 300),      # std=1
    np.random.normal(0, 1000, 300)    # std=1000
])
y_scale = ((X_unscaled[:, 0] > 0) & (X_unscaled[:, 1] > 0)).astype(int)

X_train_sc, X_test_sc, y_train_sc, y_test_sc = train_test_split(
    X_unscaled, y_scale, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sc)
X_test_scaled = scaler.transform(X_test_sc)

print(f"Feature 1 — mean: {X_train_sc[:, 0].mean():.1f}, std: {X_train_sc[:, 0].std():.1f}")
print(f"Feature 2 — mean: {X_train_sc[:, 1].mean():.1f}, std: {X_train_sc[:, 1].std():.1f}")

In [ ]:
# Compare unscaled vs scaled for all three models
model_configs = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'kNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Logistic Reg': LogisticRegression(random_state=42, max_iter=1000)
}

print(f"{'Model':<18} {'Unscaled':>10} {'Scaled':>10} {'Difference':>12}")
print('-' * 52)

for name, model in model_configs.items():
    # Unscaled
    m1 = type(model)(**model.get_params())
    m1.fit(X_train_sc, y_train_sc)
    unsc = m1.score(X_test_sc, y_test_sc)

    # Scaled
    m2 = type(model)(**model.get_params())
    m2.fit(X_train_scaled, y_train_sc)
    sc = m2.score(X_test_scaled, y_test_sc)

    print(f"{name:<18} {unsc:>10.3f} {sc:>10.3f} {sc - unsc:>+12.3f}")

# Decision Tree: ~0 difference. kNN/LogReg: large difference.

---
## Part 4: Comparing Classifiers on Imbalanced Data

Credit card fraud detection: ~0.17% of transactions are fraud.
A model that always predicts "not fraud" gets >99.8% accuracy.
That's why accuracy alone is misleading — we need precision, recall, F1.

**Precision** = of those flagged as fraud, how many actually are
**Recall** = of actual frauds, how many did we catch
**F1** = harmonic mean of precision and recall

### 4.1 Load data

In [ ]:
try:
    print("Loading creditcard.csv...")
    df = pd.read_csv('creditcard.csv')
    df = df.drop('Time', axis=1)
    X_fraud = df.drop('Class', axis=1).values
    y_fraud = df['Class'].values
    print(f"Loaded. {len(y_fraud):,} transactions, {y_fraud.sum():,} frauds ({y_fraud.mean()*100:.3f}%)")

except FileNotFoundError:
    print("creditcard.csv not found — generating simulated imbalanced data")
    from sklearn.datasets import make_classification
    X_fraud, y_fraud = make_classification(
        n_samples=10000, n_features=20, n_informative=10,
        n_classes=2, weights=[0.98, 0.02], random_state=42
    )
    print(f"Simulated. {len(y_fraud)} samples, {y_fraud.sum()} positives ({y_fraud.mean()*100:.1f}%)")

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(
    X_fraud, y_fraud, test_size=0.3, random_state=42, stratify=y_fraud
)
print(f"Train: {len(y_train_fraud)}, Test: {len(y_test_fraud)}")

### 4.2 Baseline: three classifiers

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_fraud, y_train_fraud)
    y_pred = model.predict(X_test_fraud)

    results[name] = {
        'Accuracy': accuracy_score(y_test_fraud, y_pred),
        'Precision': precision_score(y_test_fraud, y_pred, zero_division=0),
        'Recall': recall_score(y_test_fraud, y_pred, zero_division=0),
        'F1': f1_score(y_test_fraud, y_pred, zero_division=0)
    }

results_df = pd.DataFrame(results).T
print(results_df.round(3))

### 4.3 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_fraud)
    cm = confusion_matrix(y_test_fraud, y_pred)

    im = axes[i].imshow(cm, cmap='Blues')
    axes[i].set_title(f'{name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')
    axes[i].set_xticks([0, 1])
    axes[i].set_yticks([0, 1])
    axes[i].set_xticklabels(['Non-Fraud', 'Fraud'])
    axes[i].set_yticklabels(['Non-Fraud', 'Fraud'])

    for j in range(2):
        for k in range(2):
            axes[i].text(k, j, str(cm[j, k]), ha='center', va='center',
                         color='white' if cm[j, k] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.show()
# Bottom-right cell = actual fraud correctly predicted as fraud (true positives)

### 4.4 Cross-validation with multiple metrics

`cross_val_score` accepts a `scoring` parameter:
- `'accuracy'`, `'f1'`, `'precision'`, `'recall'`

In [ ]:
cv_results = {}

# Speed-up: run CV on a stratified subset
subset_size = min(30000, len(y_fraud))
idx = pd.Series(y_fraud).groupby(y_fraud, group_keys=False).apply(
    lambda s: s.sample(max(1, int(len(s) * subset_size / len(y_fraud))), random_state=42)
).index
X_cv = X_fraud[idx]
y_cv = y_fraud[idx]

for name, model in models.items():
    acc = cross_val_score(model, X_cv, y_cv, cv=3, scoring='accuracy', n_jobs=-1)
    f1 = cross_val_score(model, X_cv, y_cv, cv=3, scoring='f1', n_jobs=-1)

    cv_results[name] = {
        'Accuracy (mean)': acc.mean(),
        'Accuracy (std)': acc.std(),
        'F1 (mean)': f1.mean(),
        'F1 (std)': f1.std()
    }

cv_df = pd.DataFrame(cv_results).T
print(cv_df.round(3))
print(f"\nCV ran on {len(y_cv):,} stratified samples with cv=3 for speed.")

### 4.5 GridSearchCV — tune decision tree for F1

Optimise for F1 instead of accuracy, because on imbalanced data
accuracy is meaningless.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring='f1',
    cv=3
)
grid_search.fit(X_train_fraud, y_train_fraud)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1:  {grid_search.best_score_:.3f}")

# Compare with default tree
default_f1 = cross_val_score(
    DecisionTreeClassifier(random_state=42), X_train_fraud, y_train_fraud, cv=3, scoring='f1'
).mean()
print(f"Default CV F1: {default_f1:.3f}")

### 4.6 Tune all three models, compare

In [ ]:
param_grids = {
    'Decision Tree': {
        'max_depth': [3, 5, 7, 10],
        'min_samples_split': [2, 10, 20],
        'min_samples_leaf': [1, 5, 10]
    },
    'KNN': {
        'n_neighbors': [3, 5, 7, 9, 11],
        'weights': ['uniform', 'distance']
    },
    'Logistic Reg': {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l2'],
        'solver': ['lbfgs']
    }
}

base_models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Logistic Reg': LogisticRegression(max_iter=1000, random_state=42)
}

tuned_models = {}
tuning_results = {}

for name in base_models:
    gs = GridSearchCV(base_models[name], param_grids[name], scoring='f1', cv=3)
    gs.fit(X_train_fraud, y_train_fraud)

    tuned_models[name] = gs.best_estimator_

    y_pred = gs.best_estimator_.predict(X_test_fraud)
    tuning_results[name] = {
        'Best CV F1': gs.best_score_,
        'Test Accuracy': accuracy_score(y_test_fraud, y_pred),
        'Test Precision': precision_score(y_test_fraud, y_pred, zero_division=0),
        'Test Recall': recall_score(y_test_fraud, y_pred, zero_division=0),
        'Test F1': f1_score(y_test_fraud, y_pred, zero_division=0),
        'Best Params': str(gs.best_params_)
    }

tuning_df = pd.DataFrame(tuning_results).T
print(tuning_df[['Best CV F1', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1']].round(3))

### 4.7 Accuracy-optimised vs F1-optimised tree

What happens if we tune for accuracy instead of F1?

In [ ]:
# Tune for accuracy
grid_search_acc = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring='accuracy',
    cv=3
)
grid_search_acc.fit(X_train_fraud, y_train_fraud)

y_pred_acc = grid_search_acc.predict(X_test_fraud)
y_pred_f1 = tuned_models['Decision Tree'].predict(X_test_fraud)

comparison = pd.DataFrame({
    'Optimised for Accuracy': {
        'Accuracy': accuracy_score(y_test_fraud, y_pred_acc),
        'Precision': precision_score(y_test_fraud, y_pred_acc, zero_division=0),
        'Recall': recall_score(y_test_fraud, y_pred_acc, zero_division=0),
        'F1': f1_score(y_test_fraud, y_pred_acc, zero_division=0)
    },
    'Optimised for F1': {
        'Accuracy': accuracy_score(y_test_fraud, y_pred_f1),
        'Precision': precision_score(y_test_fraud, y_pred_f1, zero_division=0),
        'Recall': recall_score(y_test_fraud, y_pred_f1, zero_division=0),
        'F1': f1_score(y_test_fraud, y_pred_f1, zero_division=0)
    }
})

print(comparison.round(3))
# Accuracy-optimised tree may sacrifice recall for marginal accuracy gains
# F1-optimised tree catches more fraud at the cost of some false positives

### Bonus: Overfitting table — train vs test gap

In [ ]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=42
)

print(f"{'max_depth':>10} | {'Train':>8} | {'Test':>8} | {'Gap':>8}")
print('-' * 42)
for depth in [1, 2, 5, 10, 20]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_m, y_train_m)
    tr = tree.score(X_train_m, y_train_m)
    te = tree.score(X_test_m, y_test_m)
    print(f"{depth:>10} | {tr:>8.3f} | {te:>8.3f} | {tr - te:>8.3f}")

---
## Summary

### Decision Trees
```python
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
tree.fit(X_train, y_train)
tree.feature_importances_  # which features matter most
```

### Hyperparameters
| Parameter | Effect of increasing |
|-----------|---------------------|
| `max_depth` | More complex tree, risk overfitting |
| `min_samples_split` | Fewer splits, simpler tree |
| `min_samples_leaf` | Larger leaves, simpler tree |

### Scaling
| Model | Needs scaling? |
|-------|---------------|
| Decision Tree | No (only uses thresholds, order doesn't change) |
| kNN | Yes (uses distance) |
| Logistic Regression | Yes (uses gradients) |

### Metrics for Imbalanced Data
- **Accuracy** is misleading when classes are imbalanced
- **Precision** = how many flagged items are actually positive
- **Recall** = how many actual positives we caught
- **F1** = balance between precision and recall
- Use `scoring='f1'` in `GridSearchCV` / `cross_val_score` for imbalanced problems

### GridSearchCV
```python
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(model, param_grid, scoring='f1', cv=5)
grid.fit(X_train, y_train)
grid.best_params_      # best hyperparameters
grid.best_score_       # best CV score
grid.best_estimator_   # the fitted model with best params
```

### Three-Model Comparison
| | Decision Tree | kNN | Logistic Regression |
|---|---|---|---|
| Boundary shape | Rectangular | Non-linear | Linear |
| Needs scaling | No | Yes | Yes |
| Interpretable | Yes (feature importance, rules) | No | Yes (coefficients) |
| Key hyperparameter | max_depth | k | C |